# PAMCL Training on Kaggle
## Phoneme-Aware Multi-Layer Contrastive Learning for Audio Deepfake Detection

**Requirements:**
- GPU: 2x T4 (select in Settings → Accelerator)
- Dataset: ASVSpoof 2021 LA (add from Kaggle datasets)
- Code: Upload pamcl.zip as a dataset

**Resume Training:**
- Upload pamcl_checkpoints.zip as a dataset to resume from a checkpoint

## 1. Setup Environment

In [ ]:
# Check GPU availability
!nvidia-smi

In [ ]:
# Install required packages
!pip install -q g2p-en

In [ ]:
import os
import sys
import torch

# Check PyTorch and CUDA
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device count: {torch.cuda.device_count()}")
for i in range(torch.cuda.device_count()):
    print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"    Memory: {torch.cuda.get_device_properties(i).total_memory / 1e9:.1f} GB")

## 2. Extract PAMCL Code

In [ ]:
# List input datasets
!ls /kaggle/input/

In [ ]:
# Extract pamcl code
# Adjust the path based on your uploaded dataset name
PAMCL_ZIP = "/kaggle/input/pamcl-code/pamcl.zip"  # <-- CHANGE THIS to match your upload

import zipfile
import shutil

# Extract to working directory
if os.path.exists(PAMCL_ZIP):
    with zipfile.ZipFile(PAMCL_ZIP, 'r') as zip_ref:
        zip_ref.extractall('/kaggle/working/')
    print("Extracted PAMCL code")
else:
    # If uploaded as folder dataset, copy it
    PAMCL_FOLDER = "/kaggle/input/pamcl-code/pamcl"  # <-- Alternative path
    if os.path.exists(PAMCL_FOLDER):
        shutil.copytree(PAMCL_FOLDER, '/kaggle/working/pamcl')
        print("Copied PAMCL folder")
    else:
        print("ERROR: PAMCL code not found! Check your dataset upload.")

!ls /kaggle/working/

In [ ]:
# Add pamcl to Python path
sys.path.insert(0, '/kaggle/working/pamcl')
os.chdir('/kaggle/working/pamcl')
print(f"Working directory: {os.getcwd()}")

## 3. Extract Checkpoints (For Resume Training)

**If resuming from a checkpoint:**
1. Upload `pamcl_checkpoints.zip` as a Kaggle dataset
2. Update the path below
3. Run this cell to extract

In [ ]:
# ============================================
# CHECKPOINT RESUME CONFIGURATION
# ============================================

# Set to True if you want to resume from a checkpoint
RESUME_TRAINING = True  # <-- Set to True to resume, False to start fresh

# Path to your uploaded checkpoint zip
CHECKPOINT_ZIP = "/kaggle/input/pamcl-checkpoints/pamcl_checkpoints.zip"  # <-- UPDATE THIS PATH

# Alternative: Direct path to checkpoint folder if not zipped
CHECKPOINT_FOLDER = "/kaggle/input/pamcl-checkpoints/checkpoints"

# ============================================

import zipfile
import shutil

# Create checkpoints directory
os.makedirs('/kaggle/working/checkpoints', exist_ok=True)

if RESUME_TRAINING:
    print("Resume mode enabled. Looking for checkpoints...")
    
    # Try zip file first
    if os.path.exists(CHECKPOINT_ZIP):
        print(f"Found checkpoint zip: {CHECKPOINT_ZIP}")
        with zipfile.ZipFile(CHECKPOINT_ZIP, 'r') as z:
            z.extractall('/kaggle/working/checkpoints/')
        print("✓ Checkpoints extracted")
    
    # Try folder
    elif os.path.exists(CHECKPOINT_FOLDER):
        print(f"Found checkpoint folder: {CHECKPOINT_FOLDER}")
        for f in os.listdir(CHECKPOINT_FOLDER):
            src = os.path.join(CHECKPOINT_FOLDER, f)
            dst = os.path.join('/kaggle/working/checkpoints', f)
            if os.path.isfile(src):
                shutil.copy2(src, dst)
        print("✓ Checkpoints copied")
    
    else:
        print("⚠ No checkpoints found. Will start fresh training.")
        print(f"  Looked for: {CHECKPOINT_ZIP}")
        print(f"  Also tried: {CHECKPOINT_FOLDER}")
        RESUME_TRAINING = False
    
    # List available checkpoints
    if os.path.exists('/kaggle/working/checkpoints'):
        files = os.listdir('/kaggle/working/checkpoints')
        if files:
            print("\nAvailable checkpoints:")
            for f in sorted(files):
                fpath = f'/kaggle/working/checkpoints/{f}'
                if os.path.isfile(fpath):
                    size_mb = os.path.getsize(fpath) / (1024**2)
                    print(f"  - {f} ({size_mb:.1f} MB)")
else:
    print("Starting fresh training (RESUME_TRAINING = False)")

## 4. Configure Dataset Paths

**IMPORTANT:** Update these paths to match your Kaggle dataset structure.

In [ ]:
# List available datasets to find correct paths
!find /kaggle/input -maxdepth 3 -type d | head -30

In [ ]:
# Dataset configuration - PATHS FOR YOUR KAGGLE STRUCTURE
# Based on your 'asvspoof-2021' dataset

# ASVSpoof 2021 LA dataset paths
LA_AUDIO_DIR = "/kaggle/input/asvspoof-2021/ASVspoof2021_LA_eval/flac"
LA_PROTOCOL = "/kaggle/input/asvspoof-2021/LA-keys-full/keys/LA/CM/trial_metadata.txt"

# ASVSpoof 2021 DF dataset paths (for evaluation)
DF_AUDIO_DIR = "/kaggle/input/asvspoof-2021/ASVspoof2021_DF_eval_part00/flac"
DF_PROTOCOL = "/kaggle/input/asvspoof-2021/DF-keys-full/keys/DF/CM/trial_metadata.txt"

# Verify paths exist
for name, path in [("LA Audio", LA_AUDIO_DIR), ("LA Protocol", LA_PROTOCOL),
                   ("DF Audio", DF_AUDIO_DIR), ("DF Protocol", DF_PROTOCOL)]:
    exists = os.path.exists(path)
    status = "✓" if exists else "✗ NOT FOUND"
    print(f"{status} {name}: {path}")

## 5. Create Kaggle-Specific Config

In [ ]:
import yaml

# Training configuration optimized for 2x T4 GPUs
kaggle_config = {
    'model': {
        'ssl_backbone': 'facebook/wav2vec2-xls-r-300m',
        'freeze_ssl': True,
        'gradient_checkpointing': True,
        'phoneme_encoder': {
            'ctc_model': 'facebook/wav2vec2-base-960h',
            'num_phonemes': 39,
            'embedding_dim': 256
        },
        'sls': {
            'hidden_dim': 512,
            'use_sigmoid': True
        },
        'contrastive': {
            'projection_dim': 256,
            'temperature': 0.07,
            'use_phoneme_prototypes': True,
            'num_prototype_speakers': 100
        },
        'classifier': {
            'hidden_dim': 512,
            'dropout': 0.3
        }
    },
    'data': {
        'train_dir': LA_AUDIO_DIR,
        'train_protocol': LA_PROTOCOL,
        'eval_dir': DF_AUDIO_DIR,
        'eval_protocol': DF_PROTOCOL,
        'sample_rate': 16000,
        'max_audio_len': 64000,
        'min_audio_len': 16000,
        'num_workers': 4  # Kaggle can use multiple workers
    },
    'training': {
        'device': 'cuda',
        'mixed_precision': True,
        'multi_gpu': True,  # Enable multi-GPU
        'batch_size': 32,  # Per GPU, effective = 64 with 2 GPUs
        'gradient_accumulation_steps': 2,  # Effective batch = 128
        'phase1': {
            'epochs': 15,  # Research-grade
            'lr': 0.0001,
            'weight_decay': 0.01
        },
        'phase2': {
            'epochs': 20,  # Research-grade
            'lr_ssl': 0.000001,
            'lr_new': 0.0001,
            'weight_decay': 0.01
        },
        'lambda_ce': 1.0,
        'lambda_contrastive': 0.5,
        'lambda_phoneme_consistency': 0.3,
        'label_smoothing': 0.1,
        'checkpoint_dir': '/kaggle/working/checkpoints',
        'save_every_epochs': 5,
        'patience': 10,
        'min_delta': 0.001
    },
    'augmentation': {
        'enabled': True,
        'probability': 0.5,
        'add_noise': True,
        'noise_snr_range': [10, 30],
        'speed_perturb': True,
        'speed_range': [0.9, 1.1],
        'reverb': False
    },
    'logging': {
        'log_dir': '/kaggle/working/logs',
        'log_every_steps': 100,
        'tensorboard': True
    },
    'evaluation': {
        'compute_eer': True,
        'compute_auc': True
    }
}

# Save config
os.makedirs('configs', exist_ok=True)
with open('configs/kaggle.yaml', 'w') as f:
    yaml.dump(kaggle_config, f, default_flow_style=False)

print("Created configs/kaggle.yaml")
print(f"\nTraining settings:")
print(f"  Batch size per GPU: {kaggle_config['training']['batch_size']}")
print(f"  Gradient accumulation: {kaggle_config['training']['gradient_accumulation_steps']}")
print(f"  Effective batch size: {kaggle_config['training']['batch_size'] * 2 * kaggle_config['training']['gradient_accumulation_steps']}")
print(f"  Phase 1 epochs: {kaggle_config['training']['phase1']['epochs']}")
print(f"  Phase 2 epochs: {kaggle_config['training']['phase2']['epochs']}")
print(f"  Total epochs: {kaggle_config['training']['phase1']['epochs'] + kaggle_config['training']['phase2']['epochs']}")

## 6. Import and Verify Modules

In [ ]:
# Test imports
try:
    from models.pamcl_model import PAMCLModel
    from data.asvspoof_dataset import ASVSpoofDataset, collate_fn
    from data.balanced_sampler import BalancedBatchSampler
    from training.trainer import PAMCLTrainer
    from utils.helpers import load_config, set_seed, save_checkpoint
    print("✓ All modules imported successfully!")
except ImportError as e:
    print(f"✗ Import error: {e}")
    print("\nCheck that pamcl code was extracted correctly:")
    !ls -la /kaggle/working/pamcl/

## 7. Training Script with Multi-GPU Support and Resume Capability

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import logging
from pathlib import Path
import random
import sys
import time

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    stream=sys.stdout
)
logger = logging.getLogger(__name__)

def log(msg):
    """Print with immediate flush for Jupyter notebooks."""
    print(msg, flush=True)

def train_pamcl_kaggle(resume_from_checkpoint=False):
    """Main training function for Kaggle with multi-GPU support and resume capability."""
    
    log("=" * 60)
    log("PAMCL Training Starting...")
    log("=" * 60)
    
    # Load config
    log("\nLoading config...")
    config = load_config('configs/kaggle.yaml')
    log("✓ Config loaded")
    
    # Set seed for reproducibility
    set_seed(42)
    log("✓ Seed set: 42")
    
    # Device setup
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    num_gpus = torch.cuda.device_count()
    log(f"✓ Device: {device} with {num_gpus} GPU(s)")
    
    # Data setup
    log("\n" + "=" * 60)
    log("LOADING DATASET")
    log("=" * 60)
    log("(This may take a few minutes...)")
    
    audio_dir = config['data']['train_dir']
    protocol_path = config['data']['train_protocol']
    log(f"  Audio dir: {audio_dir}")
    log(f"  Protocol: {protocol_path}")
    
    # Create augmentor with proper config unpacking
    from data.augmentation import AudioAugmentor
    aug_config = config.get('augmentation', {})
    if aug_config.get('enabled', False):
        augmentor = AudioAugmentor(
            probability=aug_config.get('probability', 0.5),
            add_noise=aug_config.get('add_noise', True),
            noise_snr_range=tuple(aug_config.get('noise_snr_range', [10, 30])),
            speed_perturb=aug_config.get('speed_perturb', True),
            speed_range=tuple(aug_config.get('speed_range', [0.9, 1.1])),
            sample_rate=config['data'].get('sample_rate', 16000)
        )
    else:
        augmentor = None
    log(f"  Augmentor: {'enabled' if augmentor else 'disabled'}")
    
    log("  Creating dataset...")
    
    full_dataset = ASVSpoofDataset(
        audio_dir=audio_dir,
        protocol_path=protocol_path,
        max_audio_len=config['data'].get('max_audio_len', 64000),
        sample_rate=config['data'].get('sample_rate', 16000),
        augmentor=augmentor,
        subset_indices=None
    )
    
    log(f"✓ Total samples: {len(full_dataset)}")
    
    # Create train/val split manually
    log("Creating train/val split...")
    num_samples = len(full_dataset.samples)
    indices = list(range(num_samples))
    random.seed(42)
    random.shuffle(indices)
    
    split_idx = int(num_samples * 0.8)
    train_indices = indices[:split_idx]
    val_indices = indices[split_idx:]
    
    log(f"✓ Split: {len(train_indices)} train, {len(val_indices)} val")
    
    # Create datasets
    log("Creating training dataset...")
    train_dataset = ASVSpoofDataset(
        audio_dir=audio_dir,
        protocol_path=protocol_path,
        max_audio_len=config['data'].get('max_audio_len', 64000),
        sample_rate=config['data'].get('sample_rate', 16000),
        augmentor=augmentor,
        subset_indices=train_indices
    )
    log("✓ Training dataset ready")
    
    log("Creating validation dataset...")
    val_dataset = ASVSpoofDataset(
        audio_dir=audio_dir,
        protocol_path=protocol_path,
        max_audio_len=config['data'].get('max_audio_len', 64000),
        sample_rate=config['data'].get('sample_rate', 16000),
        augmentor=None,
        subset_indices=val_indices
    )
    log("✓ Validation dataset ready")
    
    # Count classes
    train_labels = [s['label'] for s in train_dataset.samples]
    num_real = sum(1 for l in train_labels if l == 0)
    num_fake = sum(1 for l in train_labels if l == 1)
    
    log(f"\nDataset Stats:")
    log(f"  Training: {len(train_dataset)} (real: {num_real}, fake: {num_fake})")
    log(f"  Validation: {len(val_dataset)}")
    
    # Batch size per GPU
    batch_size_per_gpu = config['training']['batch_size']
    total_batch_size = batch_size_per_gpu * num_gpus
    
    log(f"  Batch size per GPU: {batch_size_per_gpu}")
    log(f"  Total batch size: {total_batch_size}")
    
    # Balanced sampler
    log("\nCreating balanced sampler...")
    balanced_sampler = BalancedBatchSampler(
        labels=train_labels,
        batch_size=total_batch_size,
        drop_last=True
    )
    log("✓ Balanced sampler ready")
    
    # DataLoaders
    log("Creating data loaders...")
    train_loader = DataLoader(
        train_dataset,
        batch_sampler=balanced_sampler,
        num_workers=config['data'].get('num_workers', 4),
        pin_memory=True,
        collate_fn=collate_fn
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=total_batch_size,
        shuffle=False,
        num_workers=config['data'].get('num_workers', 4),
        pin_memory=True,
        collate_fn=collate_fn
    )
    
    log(f"✓ Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")
    
    # Model setup
    log("\n" + "=" * 60)
    log("INITIALIZING PAMCL MODEL")
    log("=" * 60)
    log("(Downloading model weights if needed - this may take several minutes...)")
    
    model = PAMCLModel(config)
    log("✓ Model created")
    
    # Multi-GPU with DataParallel
    if num_gpus > 1:
        log(f"Wrapping model with DataParallel ({num_gpus} GPUs)...")
        model = nn.DataParallel(model)
        # Patch for trainer compatibility
        model.get_trainable_params = model.module.get_trainable_params
        log("✓ DataParallel enabled")
    
    log("Moving model to GPU...")
    model = model.to(device)
    log("✓ Model on GPU")
    
    # Count parameters
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    log(f"\nModel Stats:")
    log(f"  Total parameters: {total_params:,}")
    log(f"  Trainable parameters: {trainable_params:,}")
    
    # Update config
    config['training']['num_real'] = 1
    config['training']['num_fake'] = 1
    
    # Create trainer
    log("\nCreating trainer...")
    
    trainer = PAMCLTrainer(
        model=model,
        config=config,
        train_loader=train_loader,
        val_loader=val_loader,
        device=device
    )
    
    log("✓ Trainer ready")
    
    # ================================================
    # RESUME FROM CHECKPOINT
    # ================================================
    start_epoch = 0
    best_eer = float('inf')
    
    checkpoint_path = '/kaggle/working/checkpoints/last_checkpoint.pt'
    
    if resume_from_checkpoint and os.path.exists(checkpoint_path):
        log("\n" + "=" * 60)
        log("RESUMING FROM CHECKPOINT")
        log("=" * 60)
        log(f"Loading: {checkpoint_path}")
        
        checkpoint = torch.load(checkpoint_path, map_location=device)
        
        # Load model state
        if isinstance(model, nn.DataParallel):
            model.module.load_state_dict(checkpoint['model_state_dict'])
        else:
            model.load_state_dict(checkpoint['model_state_dict'])
        log("✓ Model state loaded")
        
        # Load optimizer state
        trainer.optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
        log("✓ Optimizer state loaded")
        
        # Get starting epoch
        start_epoch = checkpoint['epoch'] + 1
        
        # Get best EER if available
        if 'metrics' in checkpoint and 'eer' in checkpoint.get('metrics', {}):
            best_eer = checkpoint['metrics']['eer']
        
        log(f"✓ Resuming from epoch {start_epoch}")
        log(f"  Previous best EER: {best_eer*100:.2f}%" if best_eer < float('inf') else "  No previous EER recorded")
        
        # Also try to load best model EER
        best_model_path = '/kaggle/working/checkpoints/best_model.pt'
        if os.path.exists(best_model_path):
            best_checkpoint = torch.load(best_model_path, map_location=device)
            if 'metrics' in best_checkpoint and 'eer' in best_checkpoint.get('metrics', {}):
                best_eer = best_checkpoint['metrics']['eer']
                log(f"  Best model EER: {best_eer*100:.2f}%")
    else:
        if resume_from_checkpoint:
            log("\n⚠ No checkpoint found at expected path. Starting fresh.")
        log("Starting training from epoch 0")
    
    # ================================================
    # TRAINING LOOP
    # ================================================
    total_epochs = config['training']['phase2']['epochs']
    
    log("\n" + "=" * 60)
    log(f"STARTING TRAINING: Epochs {start_epoch} to {total_epochs-1}")
    log(f"Remaining epochs: {total_epochs - start_epoch}")
    log("=" * 60 + "\n")
    
    training_start = time.time()
    
    for epoch in range(start_epoch, total_epochs):
        trainer.current_epoch = epoch
        epoch_start = time.time()
        
        log(f"\n{'='*60}")
        log(f"EPOCH {epoch + 1}/{total_epochs}")
        log(f"{'='*60}")
        
        # Training
        train_metrics = trainer.train_epoch()
        
        # Validation
        val_metrics = trainer.validate()
        
        epoch_time = time.time() - epoch_start
        
        # Print epoch summary
        log(f"\n{'='*60}")
        log(f"EPOCH {epoch + 1}/{total_epochs} COMPLETE (Time: {epoch_time:.1f}s)")
        log(f"{'='*60}")
        log(f"  Training:")
        log(f"    Loss:     {train_metrics.get('total_loss', 0):.4f}")
        log(f"    CE Loss:  {train_metrics.get('ce_loss', 0):.4f}")
        log(f"    Accuracy: {train_metrics.get('accuracy', 0)*100:.2f}%")
        log(f"    Real Acc: {train_metrics.get('real_acc', 0)*100:.2f}%  |  Fake Acc: {train_metrics.get('fake_acc', 0)*100:.2f}%")
        log(f"    Balanced: {train_metrics.get('balanced_acc', 0)*100:.2f}%")
        log(f"  Validation:")
        log(f"    Loss:     {val_metrics.get('loss', 0):.4f}")
        log(f"    EER:      {val_metrics['eer']*100:.2f}%")
        log(f"    AUC:      {val_metrics['auc']:.4f}")
        log(f"    Accuracy: {val_metrics.get('accuracy', 0)*100:.2f}%")
        
        # Save best model
        eer = val_metrics['eer']
        if eer < best_eer:
            best_eer = eer
            save_checkpoint(
                trainer.model, trainer.optimizer, epoch,
                val_metrics, '/kaggle/working/checkpoints',
                filename='best_model.pt'
            )
            log(f"  -> NEW BEST MODEL SAVED! (EER: {eer*100:.2f}%)")
        
        # Always save last checkpoint for resume
        save_checkpoint(
            trainer.model, trainer.optimizer, epoch,
            val_metrics, '/kaggle/working/checkpoints',
            filename='last_checkpoint.pt'
        )
        
        # Periodic checkpoint every 5 epochs
        if (epoch + 1) % 5 == 0:
            save_checkpoint(
                trainer.model, trainer.optimizer, epoch,
                val_metrics, '/kaggle/working/checkpoints',
                filename=f'checkpoint_epoch{epoch+1}.pt'
            )
            log(f"  -> Periodic checkpoint saved: checkpoint_epoch{epoch+1}.pt")
    
    total_time = time.time() - training_start
    
    log("\n" + "=" * 60)
    log("TRAINING COMPLETE!")
    log("=" * 60)
    log(f"Total training time: {total_time/3600:.2f} hours")
    log(f"Best EER achieved: {best_eer*100:.2f}%")
    log("\nCheckpoints saved in /kaggle/working/checkpoints/")
    
    return trainer

## 8. Run Training

In [ ]:
# Start training (set resume_from_checkpoint based on RESUME_TRAINING variable from Section 3)
trainer = train_pamcl_kaggle(resume_from_checkpoint=RESUME_TRAINING)

## 9. Save Results

In [ ]:
# List saved checkpoints
!ls -la /kaggle/working/checkpoints/

In [ ]:
# Zip checkpoints for download
import shutil

shutil.make_archive(
    '/kaggle/working/pamcl_checkpoints',
    'zip',
    '/kaggle/working/checkpoints'
)

print("Checkpoints saved to /kaggle/working/pamcl_checkpoints.zip")
!ls -lh /kaggle/working/*.zip

In [ ]:
# Create download link
from IPython.display import FileLink
display(FileLink('/kaggle/working/pamcl_checkpoints.zip'))

## 10. Quick Evaluation (Optional)

In [ ]:
# Load best model and show info
import glob

checkpoint_files = glob.glob('/kaggle/working/checkpoints/best_*.pt')
if checkpoint_files:
    best_checkpoint = checkpoint_files[0]
    print(f"Best checkpoint: {best_checkpoint}")
    
    # Load checkpoint
    checkpoint = torch.load(best_checkpoint)
    print(f"Best epoch: {checkpoint.get('epoch', 'N/A')}")
    
    if 'metrics' in checkpoint:
        metrics = checkpoint['metrics']
        print(f"Best EER: {metrics.get('eer', 0)*100:.2f}%")
        print(f"Best AUC: {metrics.get('auc', 0):.4f}")
else:
    print("No best checkpoint found yet.")

---

## Notes

### Resume Training Instructions:
1. Download `pamcl_checkpoints.zip` from Output
2. Upload it as a new Kaggle dataset (e.g., "pamcl-checkpoints")
3. Add the dataset to this notebook
4. Set `RESUME_TRAINING = True` in Section 3
5. Update `CHECKPOINT_ZIP` path if needed
6. Run all cells

### Estimated Training Time (2x T4):
- **Per epoch**: ~3.5-4 hours
- **20 epochs**: ~70-80 hours (split across sessions)

### Tips:
1. **Save checkpoints frequently** - Download zip every few epochs
2. **Use Kaggle's "Save & Run All"** - Runs in background
3. **Monitor Output panel** - Watch for checkpoints
4. **Resume seamlessly** - Upload checkpoint zip and continue

### Troubleshooting:
- If OOM error: Reduce `batch_size` in config
- If import errors: Check that pamcl.zip was extracted correctly
- If dataset not found: Update paths in Section 4
- If resume fails: Check checkpoint path in Section 3